In [1]:
ConnectString = "mysql -h database-1.cfvrvxxcdsku.ca-west-1.rds.amazonaws.com -P 3306 -u admin -p --ssl-mode=VERIFY_IDENTITY --ssl-ca=./global-bundle.pem"

host=ConnectString[9:60]
user='admin'
password='Data608-Project'
database='' # Optional: leave out to create DB first


In [2]:
#!pip install mysql-connector-python --break-system-packages

In [3]:
import mysql.connector
from mysql.connector import Error

In [4]:
connection = mysql.connector.connect(
    host=host,
    user=user,
    password=password,
    database='' # Optional: leave out to create DB first
   )

if connection.is_connected():
    db_info = info = connection.server_info
    print(f"Connected to MySQL Server version: {db_info}")
    cursor = connection.cursor()


Connected to MySQL Server version: 8.4.7


In [5]:

# 2. Create a cursor
cursor = connection.cursor()

# 3. Execute the CREATE DATABASE command
# Use "IF NOT EXISTS" to prevent errors if the DB already exists
cursor.execute("CREATE DATABASE IF NOT EXISTS CHESSBOT")

print("Database created successfully!")

# 4. (Optional) Switch to the new database to start using it
#cursor.execute("USE my_new_database")
# Alternatively, re-assign the database property directly:
# mydb.database = 'my_new_database'


Database created successfully!


In [6]:
# 1. Execute the command
cursor.execute("SHOW DATABASES")

# 2. Fetch and print the results
print("Available Databases:")
for (db_name,) in cursor:
    print(f"- {db_name}")

Available Databases:
- CHESSBOT
- information_schema
- mysql
- performance_schema
- sys


In [7]:
#select database to use
cursor = connection.cursor()
cursor.execute("USE CHESSBOT")
print("Selected CHESSBOT database")

Selected CHESSBOT database


In [8]:
#cursor.execute("DROP TABLE IF EXISTS files")
#cursor.execute("DROP TABLE IF EXISTS session")
#cursor.execute("DROP TABLE IF EXISTS moves")
#cursor.execute("DROP TABLE IF EXISTS games")

In [9]:
# --- CREATE: Create a table ---
games_table_name = 'games'
moves_table_name = 'moves'
files_table_name = 'files'

gamestableexists = False
movestableexists = False
filestableexists = False

# 1. Execute the command to check if table exists
cursor.execute("SHOW TABLES")

# 2. Fetch and print the results, capture table exists
print("Available Tables:")
for (db_name,) in cursor:
    print(f"- {db_name}")
    if(db_name == games_table_name):
        gamestableexists = True
        print ("-------------")
        print ("table: ",games_table_name," already exists so will not recreate")
    if(db_name == moves_table_name):
        movestableexists = True
        print ("-------------")
        print ("table: ",moves_table_name," already exists so will not recreate")
    if(db_name == files_table_name):
        filestableexists = True
        print ("-------------")
        print ("table: ",files_table_name," already exists so will not recreate")


if gamestableexists == False:   # create games table
    print ("-------------")
    print ("table: ",games_table_name," does not exists")
    CreateTable = "CREATE TABLE " + games_table_name + "(id INT AUTO_INCREMENT PRIMARY KEY,Site VARCHAR(50),White_ELO INT,Black_ELO INT,Opening VARCHAR(100))"
    print(CreateTable)
    cursor.execute(CreateTable)
    print("Table ",games_table_name, " created.")    

if movestableexists == False:   # create moves table
    print ("-------------")
    print ("table: ",moves_table_name," does not exists")
    CreateTable2 = "CREATE TABLE " + moves_table_name + "(id INT AUTO_INCREMENT PRIMARY KEY,Set5 VARCHAR(75),Set10 VARCHAR(75),Set15 VARCHAR(75),Set20 VARCHAR(75),Set30 VARCHAR(150),Set40 VARCHAR(150),Set70 VARCHAR(450),Set_Remain VARCHAR(450),BlackELO INT,Game_id INT,FOREIGN KEY (Game_id) REFERENCES games(id))"
    print(CreateTable2)
    cursor.execute(CreateTable2)
    print("Table ",moves_table_name, " created.")    

    ### create Move indexes
    cursor.execute("CREATE INDEX idx_set5 ON moves(Set5)")
    cursor.execute("CREATE INDEX idx_set10 ON moves(Set10)")
    cursor.execute("CREATE INDEX idx_set15 ON moves(Set15)")
    cursor.execute("CREATE INDEX idx_set20 ON moves(Set20)")
    cursor.execute("CREATE INDEX idx_set30 ON moves(Set30)")
    print ("-------------")
    print ("Moves table indexes are created")

if filestableexists == False:   # create files table
    print ("-------------")
    print ("table: ",files_table_name," does not exists")
    CreateTable3 = "CREATE TABLE " + files_table_name + "(id INT AUTO_INCREMENT PRIMARY KEY,filename VARCHAR(100),DateTimeLoaded VARCHAR(50),NumGamesTotal INT,NumGamesIngested INT,Month_value VARCHAR(3),Year_value VARCHAR(5),File_size VARCHAR(50))"
    print(CreateTable3)
    cursor.execute(CreateTable3)
    print("Table ",files_table_name, " created.")    


Available Tables:
- files
-------------
table:  files  already exists so will not recreate
- games
-------------
table:  games  already exists so will not recreate
- moves
-------------
table:  moves  already exists so will not recreate


In [10]:
#check games table

cursor.execute("DESCRIBE "+games_table_name)

# Fetch and print the headers
print(f"{'Field':<20} {'Type':<15} {'Null':<10} {'Key':<5}")
print("-" * 50)

for row in cursor.fetchall():
    print(f"{row[0]:<20} {row[1]:<15} {row[2]:<10} {row[3]:<5}")

Field                Type            Null       Key  
--------------------------------------------------
id                   int             NO         PRI  
Site                 varchar(50)     YES             
White_ELO            int             YES             
Black_ELO            int             YES             
Opening              varchar(100)    YES             


In [11]:
#check moves table

cursor.execute("DESCRIBE "+moves_table_name)

# Fetch and print the headers
print(f"{'Field':<20} {'Type':<15} {'Null':<10} {'Key':<5}")
print("-" * 50)

for row in cursor.fetchall():
    print(f"{row[0]:<20} {row[1]:<15} {row[2]:<10} {row[3]:<5}")

Field                Type            Null       Key  
--------------------------------------------------
id                   int             NO         PRI  
Set5                 varchar(75)     YES        MUL  
Set10                varchar(75)     YES        MUL  
Set15                varchar(75)     YES        MUL  
Set20                varchar(75)     YES        MUL  
Set30                varchar(150)    YES        MUL  
Set40                varchar(150)    YES             
Set70                varchar(450)    YES             
Set_Remain           varchar(450)    YES             
BlackELO             int             YES             
Game_id              int             YES        MUL  


In [12]:
#check files table

cursor.execute("DESCRIBE "+files_table_name)

# Fetch and print the headers
print(f"{'Field':<20} {'Type':<15} {'Null':<10} {'Key':<5}")
print("-" * 50)

for row in cursor.fetchall():
    print(f"{row[0]:<20} {row[1]:<15} {row[2]:<10} {row[3]:<5}")

Field                Type            Null       Key  
--------------------------------------------------
id                   int             NO         PRI  
filename             varchar(100)    YES             
DateTimeLoaded       varchar(50)     YES             
NumGamesTotal        int             YES             
NumGamesIngested     int             YES             
Month_value          varchar(3)      YES             
Year_value           varchar(5)      YES             
File_size            varchar(50)     YES             


In [13]:
# --- READ: Fetch the record count of games table ---
cursor.execute(f"SELECT COUNT(*) FROM {games_table_name}")
count = cursor.fetchone()[0]
print(f"{games_table_name} table has entries: {count}")


games table has entries: 3720934


In [14]:
# --- READ: Fetch the record count of moves table ---
cursor.execute(f"SELECT COUNT(*) FROM {moves_table_name}")
count = cursor.fetchone()[0]
print(f"{moves_table_name} table has entries: {count}")


moves table has entries: 3720933


In [15]:
# --- READ: Fetch the record count of files table ---
cursor.execute(f"SELECT COUNT(*) FROM {files_table_name}")
count = cursor.fetchone()[0]
print(f"{files_table_name} table has entries: {count}")


files table has entries: 27


In [16]:
#Michael's sandbox..
#INSERT

insert_query = "INSERT INTO " + games_table_name + "(Site, White_ELO,Black_ELO,Opening) VALUES (%s, %s, %s, %s)"
cursor.execute(insert_query, ("https://lichess.org/gx3qb4ur", 1155,1096,"King's Indian Attack"))
connection.commit()  # Required to save changes
InsertedRecord = cursor.lastrowid
print(f"Record inserted. ID: {InsertedRecord}")


Record inserted. ID: 3720944


In [17]:
#Michael's sandbox..
#READ

sqlstatement = "SELECT * FROM files"
cursor.execute(sqlstatement)
records = cursor.fetchall()
print(f"Total rows: {cursor.rowcount}")
for row in records:
    print("---------------")
    for value in row:
        print(str(value))



Total rows: 27
---------------
1
lichess_db_standard_rated_2013-01.pgn.zst
2026-03-26 21:18:52.517334
86856
34476
01
2013
17761302
---------------
2
lichess_db_standard_rated_2013-02.pgn.zst
2026-03-26 21:28:51.233815
88469
35492
02
2013
18151480
---------------
3
lichess_db_standard_rated_2014-01.pgn.zst
2026-03-26 22:10:19.935720
522995
174475
01
2014
111440294
---------------
4
lichess_db_standard_rated_2018-01.pgn.zst
2026-03-26 22:53:59.400619
17888152
17244
01
2018
5472079625
---------------
6
lichess_db_standard_rated_2018-02.pgn.zst
2026-03-27 01:11:19.658996
17330529
16185
02
2018
5282843923
---------------
7
lichess_db_standard_rated_2013-03.pgn.zst
2026-03-27 17:40:57.811425
114486
44149
03
2013
23590691
---------------
8
lichess_db_standard_rated_2013-04.pgn.zst
2026-03-27 17:59:52.921226
114320
43551
04
2013
23299559
---------------
9
lichess_db_standard_rated_2013-05.pgn.zst
2026-03-27 18:12:41.076555
129956
49594
05
2013
26545457
---------------
10
lichess_db_standard_ra

In [27]:
#Michael's sandbox..
#READ

sqlstatement = "SELECT * FROM moves WHERE  SET5 LIKE '1. e2e4 e7e5 2. g1f3 b8c6 3. f1b5 d7d6%' ORDER BY id ASC LIMIT 0,15"
cursor.execute(sqlstatement)
records = cursor.fetchall()
print(f"Total rows: {cursor.rowcount}")
for row in records:
    print("---------------")
    for value in row:
        print(str(value))



Total rows: 15
---------------
180
1. e2e4 e7e5 2. g1f3 b8c6 3. f1b5 d7d6 4. d2d4 a7a6 5. b5a4 b7b5
6. a4b3 e5d4 7. f3d4 c6d4 8. d1d4 g8f6 9. c1g5 f8e7 10. g5f6 e7f6
11. d4d5 e8g8 12. d5a8 f6b2 13. e1g1 b2a1 14. b1d2 a1c3 15. d2f3 d8f6
16. a8d5 f6g6 17. f3d4 c8h3 18. g2g3 h3f1 19. g1f1 c3d4 20. d5d4 f8e8
21. d4d5 g6e4 22. d5f7 g8h8 23. f7c7 e4h1
None
None
None
1566
181
---------------
195
1. e2e4 e7e5 2. g1f3 b8c6 3. f1b5 d7d6 4. c2c3 c8d7 5. d2d4 g8f6
6. b1d2 f8e7 7. e1g1 e8g8 8. h2h3 a7a6 9. b5a4 b7b5 10. a4c2 a8b8
11. f1e1 b5b4 12. d2f1 a6a5 13. f1g3 e5d4 14. c3d4 f8e8 15. e4e5 d6e5
16. d4e5 c6e5 17. f3e5 e7d6 18. c1f4 b8b5 19. e5d7 d8d7 20. f4d6 c7d6
21. e1e8 f6e8 22. d1d3 e8f6 23. c2a4 b5d5 24. d3d5 d7a4 25. d5a8 f6e8 26. a1e1 g8f8 27. g3f5 a4d7 28. f5d6 g7g6 29. d6e8 d7e8 30. e1e8 f8g7
31. e8g8 g7h6 32. h3h4 f7f5 33. a8d8 h6h5 34. d8g5
None
None
1647
196
---------------
331
1. e2e4 e7e5 2. g1f3 b8c6 3. f1b5 d7d6 4. b5c6 b7c6 5. e1g1 g8f6
6. d2d3 c8g4 7. c1g5 f8e7 8. b1d2 h7h6 9. 

In [19]:
#Michael's sandbox..
#UPDATE

update_query = "UPDATE games SET Opening = %s WHERE id = %s"
cursor.execute(update_query, ("Some new death method", InsertedRecord))
connection.commit()
print("Record updated.")


Record updated.


In [20]:
#Michael's sandbox..
#DELETE

delete_query = "DELETE FROM games WHERE id = %s"
cursor.execute(delete_query, (InsertedRecord,))
connection.commit()
print("Record deleted.")


Record deleted.


In [21]:

sqlstatement = """SELECT table_schema "CHESSBOT", 
ROUND(SUM(data_length + index_length) / 1024 / 1024 / 1024, 2) "Size (GB)" 
FROM information_schema.tables 
GROUP BY table_schema;"""

cursor.execute(sqlstatement)
records = cursor.fetchall()
print(f"Total rows: {cursor.rowcount}")
for row in records:
    print("---------------")
    for value in row:
        print(str(value))



Total rows: 5
---------------
CHESSBOT
4.36
---------------
information_schema
0.00
---------------
mysql
0.01
---------------
performance_schema
0.00
---------------
sys
0.00
